In [ ]:
%pip install transformers torch torchvision torchaudio numpy pandas tqdm matplotlib huggingface_hub datasets evaluate scikit-learn accelerate rouge-score bert-score sentence-transformers

In [2]:
import sys
import os

# Path to the folder you want to add
subfolder_path = os.path.join(os.getcwd(), "byt5_generator")

# Add it to sys.path
if subfolder_path not in sys.path:
    sys.path.append(subfolder_path)

In [3]:
import os
import argparse
import pandas as pd
import transformers

from transformers import AutoModelForSeq2SeqLM
from byt5_generator.seeding import enforce_reproducibility
from byt5_generator.dataset import prepare_datasets
from byt5_generator.train import train_seq2seq, evaluate_seq2seq

transformers.logging.set_verbosity_error()

/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:

from collections import Counter


model_name = "google/byt5-base"
output_dir = f"byt5_training_results"
epochs = 3

enforce_reproducibility(42)

# ==== PREPARE DATASETS ====
train_set, val_set, test_set, tokenizer = prepare_datasets(model_name)
train_splits = Counter(train_set["answerable"])
print(f"Sampled Train dataset class split: {train_splits}, with a total of {len(train_set)} samples")
val_splits = Counter(val_set["answerable"])
print(f"Sampled Val dataset class split: {val_splits}, with a total of {len(val_set)} samples")
test_splits = Counter(test_set["answerable"])
print(f"Sampled Test dataset class split: {test_splits}, with a total of {len(test_set)} samples")

# ==== MODEL ====
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# ==== TRAIN MODEL ====
model, tokenizer, step_logs = train_seq2seq(
    model, train_set, val_set, tokenizer, epochs, output_dir
)

# ==== EVALUATE MODEL ====
test_logs = evaluate_seq2seq(model, tokenizer, test_set, output_dir)

Found 3 mislabelled instances


Creating CSV from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 789.14ba/s]
Creating CSV from Arrow format: 0ba [00:00, ?ba/s]

[2025-10-27 16:07:34,658] - [INFO] - Train dataset has total of 39 samples
[2025-10-27 16:07:34,658] - [INFO] - Validation dataset has total of 5 samples
[2025-10-27 16:07:34,659] - [INFO] - Test dataset dataset has total of 87 samples


Sampled Train dataset class split: Counter({False: 35, True: 4}), with a total of 39 samples
Sampled Val dataset class split: Counter({False: 4, True: 1}), with a total of 5 samples
Sampled Test dataset class split: Counter({False: 80, True: 7}), with a total of 87 samples
Question: భరత్ అనే నేను చిత్ర నిర్మాత ఎవరు?
Context: Bharat Ane Nenu is a Telugu movie released in 2018 directed by Koratala Siva. Mahesh Babu is playing the lead role in this movie produced by DVV Danayya under the banner of DVV Entertainments and Kaira Advani is playing the lead role. Devi Sri Prasad provided the music. Bharat Ram (Mahesh Babu) loves to learn new things. That's why he continues to do degrees at London Oxford University. At such a time the death of his father Raghava (Sarath Kumar) turns his life around. Political guru Varada (Prakash Raj) makes Bharath the chief minister to avoid a split in the party after the death of Raghava, who founded the Navodayam party with the intention of serving the peopl

/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Error: command buffer exited with error status.
	The Metal Performance Shaders operations encoded on it may not have completed.
	Error: 
	(null)
	Insufficient Memory (00000008:kIOGPUCommandBufferCallbackErrorOutOfMemory)
	<AGXG16GFamilyCommandBuffer: 0x3275efa70>
    label = <none> 
    device = <AGXG16GDevice: 0x1167b4e00>
        name = Apple M4 
    commandQueue = <AGXG16GFamilyCommandQueue: 0x114068200>
        label = <none> 
        device = <AGXG16GDevice: 0x1167b4e00>
            name = Apple M4 
    retainedReferences = 1


Predicted:  భరత్ అనే నేను చిత్ర నిర్మాత ఎవరు?
భరత్ అనే నేను 
Target: డివివి దానయ్య
F1 Score: 0.46153846153846156
Answerable: True
Predicted: 
యానాం యొక్క విస్తీర్ణం ఎంత?
యానాం యొక్క విస్తీ
Target: 30 చ.కి.మీ
F1 Score: 0.2962962962962963
Answerable: False
{'eval_loss': 25.03229331970215, 'eval_f1_overall': 0.26941641393254295, 'eval_f1_answerable': 0.46153846153846156, 'eval_f1_unanswerable': 0.2213859020310633, 'eval_bertscore_overall': 0.6110981106758118, 'eval_bertscore_answerable': 0.6237667798995972, 'eval_bertscore_unanswerable': 0.6079309433698654, 'eval_semantic_overall': 0.5390521287918091, 'eval_semantic_answerable': 0.796689510345459, 'eval_semantic_unanswerable': 0.4746427536010742, 'eval_runtime': 23.4392, 'eval_samples_per_second': 0.213, 'eval_steps_per_second': 0.128, 'epoch': 0.1}


Error: command buffer exited with error status.
	The Metal Performance Shaders operations encoded on it may not have completed.
	Error: 
	(null)
	Insufficient Memory (00000008:kIOGPUCommandBufferCallbackErrorOutOfMemory)
	<AGXG16GFamilyCommandBuffer: 0x467ea0600>
    label = <none> 
    device = <AGXG16GDevice: 0x1167b4e00>
        name = Apple M4 
    commandQueue = <AGXG16GFamilyCommandQueue: 0x114068200>
        label = <none> 
        device = <AGXG16GDevice: 0x1167b4e00>
            name = Apple M4 
    retainedReferences = 1
/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Error: command buffer exited with error status.
	The Metal Performance Shaders operations encoded on it may not have completed.
	Error: 
	(null)
	Insufficient Memory (00000008:kIOGPUCommandBufferCallbackErrorOutOfMem

KeyboardInterrupt: 